In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # what is meesage place holder ??
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory #in built memory in langchain --
from langchain_core.runnables.history import RunnableWithMessageHistory # wait for 15 mint
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage # by default your LLM try --> input --> human or AI meeage
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from pydantic import BaseModel, Field
from typing import List


# BaseChatMessage History ==> add_message, clear message

# class --> inher BaseChatMessage ( abstract class) ==> name of method == you need to write the llogic ???


In [2]:
load_dotenv()

True

In [14]:
mesage = ['H:AI','H:AI','H:AI','H:AI','H:AI']
if len(mesage)>4:
    print('isndie if')
    mesage= mesage[3:]
    

# mesage==?LLM

mesage

 



isndie if


['H:AI', 'H:AI']

In [30]:
class WindowChatMessageHistory(BaseChatMessageHistory,BaseModel):
    messages : List[BaseMessage]=Field(default_factory=list)
    k: int = Field(default=6) # keep last 6 message ( k/2 turn )
    def add_messages(self,messages):
        self.messages.extend(messages)
        if len(self.messages)>self.k:
            dropped = len(self.messages) - self.k # 10===10-6=4( remove 4 meesage ==> message lent always 6) 7-6==>1
            self.messages= self.messages[-self.k:]
    def clear(self):
        self.messages=[]

In [31]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [32]:
WINDOW_K = 4  # keep last 4 messages = 2 full turns (human + AI each)

prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify: Billing / Technical / General. Priority: High / Medium / Low.
Remember customer details within the conversation."""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt | llm | StrOutputParser()

In [33]:
window_store = {}

def get_session_history(session_id):
    if session_id not in window_store:
        window_store[session_id] = WindowChatMessageHistory(k=WINDOW_K)
    return window_store[session_id]

In [35]:
window_store

{'test': WindowChatMessageHistory(messages=[], k=4)}

In [34]:
get_session_history('test')

WindowChatMessageHistory(messages=[], k=4)

In [36]:
window_assistant = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

In [37]:
cfg = {"configurable": {"session_id": "window_john"}}


In [38]:
window_assistant.invoke({"input":"Billing complaint from John — invoice #1042, overcharged $250."},
                        config=cfg)

'Classification: Billing  \nPriority: High  \n\nCustomer Details: John; Invoice #1042; Overcharged $250.'

In [39]:
window_store

{'test': WindowChatMessageHistory(messages=[], k=4),
 'window_john': WindowChatMessageHistory(messages=[HumanMessage(content='Billing complaint from John — invoice #1042, overcharged $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Classification: Billing  \nPriority: High  \n\nCustomer Details: John; Invoice #1042; Overcharged $250.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], k=4)}

In [40]:
turns = [
    "Billing complaint from John — invoice #1042, overcharged $250.",
    "What category is his complaint?",
    "What is the SLA for billing?",         # ← Turn 3 — Turn 1 gets dropped after this
    "Can you remind me who made the original complaint?",  # ← CRITICAL: Turn 1 is now GONE
    "Draft a 2-line apology email for him.",
]


for i, turn in enumerate(turns, 1):
    response = window_assistant.invoke({"input": turn}, config=cfg)
    hist = window_store["window_john"]
    print(response)

Classification: Billing  
Priority: High  

Customer Details: John; Invoice #1042; Overcharged $250.
The complaint from John falls under the category of **Billing**.
The typical Service Level Agreement (SLA) for billing inquiries is usually 48 to 72 hours for resolution or response. However, specific SLAs may vary depending on the company or service provider. Please refer to your company's policy for the exact SLA details.
The original complaint was made by a customer named John.
Subject: Apology for Inconvenience

Dear John, I sincerely apologize for any inconvenience caused regarding your recent complaint. We are committed to resolving your issue promptly.


In [44]:
window_assistant.invoke({"input":"Hi how are you."},
                        config=cfg)

"I'm here to assist you! How can I help you today?"

In [45]:
window_store['window_john'].messages

[HumanMessage(content='Draft a 2-line apology email for him.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Subject: Apology for Inconvenience\n\nDear John, I sincerely apologize for any inconvenience caused regarding your recent complaint. We are committed to resolving your issue promptly.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Hi how are you.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm here to assist you! How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
# tool ijtegration ===>
# tool i tegrations == LLM will any api ==> db => files 

LL -. tool --> memneory